# 01 — Pandas Core Concepts

Series/DataFrame internals, the index, dtypes, vectorization, and the views-vs-copies footgun — the vocabulary every data engineer interview opens with, and the single most common source of silent pandas bugs.

In [1]:
import pandas as pd
import numpy as np

pd.__version__, np.__version__

('3.0.5', '2.4.6')

## 1. Series and DataFrame

- A **Series** is a 1-D labeled array: values (a NumPy array or extension array) + an **index** (the labels). A DataFrame is a dict of Series sharing one index — each column can have its own dtype.
- The **index** is not just row numbers — it's how pandas aligns data across operations (arithmetic between two Series aligns on index labels first, not position). This is a frequent source of surprise bugs when two DataFrames don't share the index you assumed.

In [2]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
s2 = pd.Series([1, 2, 3], index=["b", "c", "d"])

print(s + s2)   # aligns on index labels -> 'a' and 'd' become NaN, not an error

a     NaN
b    21.0
c    32.0
d     NaN
dtype: float64


## 2. dtypes matter — pandas is not "just Python objects"

Numeric columns are backed by contiguous NumPy arrays (`int64`, `float64`) — fast, vectorized C operations. A column of mixed types, or strings without the `"string"` extension dtype, becomes dtype `object` — an array of Python object pointers, which loses vectorization speed and pandas-level null-handling guarantees.

**Interview point:** "Check `df.dtypes` early — an unexpectedly `object` column (e.g. a numeric column with a stray `'N/A'` string mixed in) silently falls back to slow, per-element Python operations."

In [3]:
df = pd.DataFrame({
    "user_id": [1, 2, 3],
    "amount": [10.5, 20.25, None],
    "plan": ["pro", "free", "pro"],
    "signup_date": pd.to_datetime(["2024-01-01", "2024-02-15", "2024-03-20"]),
})
df.dtypes

user_id                 int64
amount                float64
plan                      str
signup_date    datetime64[us]
dtype: object

## 3. Vectorization vs. Python loops

The single biggest pandas performance lesson: operations on whole columns (`df["amount"] * 1.1`) run in compiled C/NumPy code across the entire array at once. A Python-level loop (`for i in range(len(df))`, or `.apply()` with a Python function) pays Python's per-element interpreter overhead on every row. Always reach for a vectorized expression, a built-in Series/DataFrame method, or NumPy, before writing a loop or `.apply()`.

In [4]:
import time

big = pd.DataFrame({"x": np.random.rand(200_000)})

start = time.perf_counter()
vectorized = big["x"] * 2 + 1
vector_time = time.perf_counter() - start

start = time.perf_counter()
looped = big["x"].apply(lambda v: v * 2 + 1)
apply_time = time.perf_counter() - start

print(f"vectorized: {vector_time*1000:.2f} ms")
print(f"apply:      {apply_time*1000:.2f} ms")
print(f"apply is ~{apply_time/vector_time:.0f}x slower")

vectorized: 3.23 ms
apply:      32.86 ms
apply is ~10x slower


## 4. Views vs. copies — `SettingWithCopyWarning`

Some pandas operations return a **view** (shares memory with the original) and some return a **copy** — and which one you get is not always obvious from the syntax alone (it can depend on the dtype layout at runtime). Writing into the result of a chained-indexing operation like `df[df.x > 0]["y"] = 1` may silently do nothing to `df`, or emit `SettingWithCopyWarning` because pandas can't tell if you meant to modify a temporary or the original.

**The fix, always:** use a single `.loc[]` call for combined row+column selection when assigning, and call `.copy()` explicitly when you intend to build an independent DataFrame from a slice.

In [5]:
# Risky: chained indexing -- may or may not mutate df, and warns
# df[df["plan"] == "pro"]["amount"] = 0          # AVOID

# Correct: single .loc call, unambiguous
df.loc[df["plan"] == "pro", "amount"] = 0
df

,user_id,amount,plan,signup_date
0,1,0.00,pro,2024-01-01
1,2,20.25,free,2024-02-15
2,3,0.00,pro,2024-03-20


In [6]:
# When you want an independent copy to modify without touching the original:
pro_only = df[df["plan"] == "pro"].copy()
pro_only["amount"] = 999          # safe -- pro_only is a real independent copy
print(pro_only)
print(df)                          # untouched

   user_id  amount plan signup_date
0        1     999  pro  2024-01-01
2        3     999  pro  2024-03-20
   user_id  amount  plan signup_date
0        1    0.00   pro  2024-01-01
1        2   20.25  free  2024-02-15
2        3    0.00   pro  2024-03-20


## 5. Interview Q&A

1. **"Why is my numeric column dtype `object` instead of `float64`?"** — it contains at least one non-numeric value (a stray string, or mixed types); pandas falls back to Python objects. Check with `df.dtypes` and `df["col"].apply(type).value_counts()`.
2. **"Why did assigning into a filtered slice not update my DataFrame?"** — chained indexing (`df[mask]["col"] = x`) can operate on a temporary copy pandas created for the first `[]`; use `df.loc[mask, "col"] = x` instead.
3. **"Why is `.apply()` slow compared to a vectorized expression?"** — `.apply()` calls a Python function once per row/element, paying interpreter overhead each time; a vectorized op runs compiled NumPy/C code over the whole array in one call.
4. **"What does the index actually do besides labeling rows?"** — it drives alignment (arithmetic/joins match on index labels), and is what makes `.loc[]` label-based lookups O(1)-ish when the index is unique and sorted/hashed appropriately.

## Summary

- The index drives alignment, not just display — mismatched indexes between two Series/DataFrames silently produce `NaN`s.
- Check `df.dtypes`; an `object` column where you expect numeric/string data is a common silent-slowdown/bug source.
- Vectorize; avoid `.apply()`/loops unless the logic genuinely can't be expressed as a column operation.
- Use `.loc[mask, col] = value` for assignment, `.copy()` when you want an independent DataFrame from a slice.
- Next: `02_data_io_and_cleaning.ipynb`.